# 08 — Large Files & Memory Optimization

Reading multi-GB files naively will OOM-kill your process. This notebook covers chunked
reading, dtype specification at load time, memory profiling, and the Arrow backend.

In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile
import time

TMPDIR = tempfile.mkdtemp()

# Generate a large CSV to simulate big-data scenarios
np.random.seed(42)
n = 500_000

big_df = pd.DataFrame({
    'transaction_id':  range(1, n + 1),
    'customer_id':     np.random.randint(1, 50000, n),
    'product_id':      np.random.randint(1, 1000, n),
    'category':        np.random.choice(['Electronics', 'Clothing', 'Food', 'Books', 'Sports'], n),
    'amount':          np.round(np.random.uniform(10, 5000, n), 2),
    'quantity':        np.random.randint(1, 10, n),
    'city':            np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai',
                                          'Kolkata', 'Hyderabad', 'Ahmedabad'], n),
    'payment_method':  np.random.choice(['Card', 'UPI', 'Cash', 'EMI'], n),
    'date':            pd.date_range('2022-01-01', periods=n, freq='min').strftime('%Y-%m-%d'),
    'is_returned':     np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

csv_path = os.path.join(TMPDIR, 'transactions.csv')
big_df.to_csv(csv_path, index=False)

file_size_mb = os.path.getsize(csv_path) / 1024**2
print(f"Generated {n:,} rows | File: {file_size_mb:.1f} MB")

## 1. Naive Read — Baseline Memory

In [ ]:
start = time.perf_counter()
df_naive = pd.read_csv(csv_path)
t_naive = time.perf_counter() - start

mem_naive = df_naive.memory_usage(deep=True).sum() / 1024**2
print(f"Read time:    {t_naive:.2f}s")
print(f"Memory usage: {mem_naive:.1f} MB")
print(f"Dtypes:")
print(df_naive.dtypes)

## 2. Optimized Read — Specify dtypes at Load Time

**Always specify dtypes in `read_csv()` for production pipelines** — avoids the dtype-inference
pass and dramatically reduces memory.

In [ ]:
dtype_map = {
    'transaction_id': 'int32',
    'customer_id':    'int32',
    'product_id':     'int16',
    'category':       'category',
    'amount':         'float32',
    'quantity':       'int8',
    'city':           'category',
    'payment_method': 'category',
    'is_returned':    'int8',
}

start = time.perf_counter()
df_opt = pd.read_csv(
    csv_path,
    dtype=dtype_map,
    parse_dates=['date'],
)
t_opt = time.perf_counter() - start

mem_opt = df_opt.memory_usage(deep=True).sum() / 1024**2
print(f"Optimized read time: {t_opt:.2f}s")
print(f"Optimized memory:    {mem_opt:.1f} MB")
print(f"Naive memory:        {mem_naive:.1f} MB")
print(f"Reduction:           {(1 - mem_opt/mem_naive):.1%}")

In [ ]:
# Column-level memory breakdown
mem_by_col = df_opt.memory_usage(deep=True).sort_values(ascending=False)
print("Memory per column (MB):")
for col, mem in mem_by_col.items():
    print(f"  {col:<20} {mem/1024**2:.2f} MB")

## 3. usecols — Read Only What You Need

In [ ]:
# Only read columns needed for analysis — huge memory saving
needed_cols = ['transaction_id', 'customer_id', 'category', 'amount', 'date']

start = time.perf_counter()
df_narrow = pd.read_csv(
    csv_path,
    usecols=needed_cols,
    dtype={
        'transaction_id': 'int32',
        'customer_id':    'int32',
        'category':       'category',
        'amount':         'float32',
    },
    parse_dates=['date'],
)
t_narrow = time.perf_counter() - start
mem_narrow = df_narrow.memory_usage(deep=True).sum() / 1024**2

print(f"Narrow read time: {t_narrow:.2f}s")
print(f"Memory:           {mem_narrow:.1f} MB")
print(f"vs Full optimized: {mem_opt:.1f} MB")

## 4. Chunked Reading — Process Without Loading Entire File

In [ ]:
# Use chunksize to process file in pieces
CHUNK_SIZE = 50_000

# Pattern: aggregate across chunks
category_totals = {}

for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, dtype=dtype_map):
    # Process each chunk independently
    chunk_agg = chunk.groupby('category')['amount'].sum()
    for cat, total in chunk_agg.items():
        category_totals[cat] = category_totals.get(cat, 0) + total

result = pd.Series(category_totals, name='total_revenue').sort_values(ascending=False)
print("Revenue by category (chunked computation):")
print(result.map(lambda x: f"{x:,.0f}"))

In [ ]:
# Pattern: collect matching rows across chunks (filter-then-collect)
high_value_chunks = []

for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE,
                         dtype=dtype_map, usecols=['transaction_id', 'customer_id', 'amount', 'category']):
    high_value_chunks.append(chunk[chunk['amount'] > 4000])

high_value = pd.concat(high_value_chunks, ignore_index=True)
print(f"High-value transactions (>4000): {len(high_value):,}")
print(high_value.head())

## 5. TextFileReader Object — More Control

In [ ]:
# get_chunk() gives manual control — process only what you need
reader = pd.read_csv(csv_path, chunksize=10000, dtype=dtype_map)

first_chunk = reader.get_chunk()   # process first chunk
second_chunk = reader.get_chunk()  # process second chunk

print(f"First chunk shape:  {first_chunk.shape}")
print(f"Second chunk shape: {second_chunk.shape}")

reader.close()  # always close the reader

## 6. memory_usage() — Profile Your DataFrame

In [ ]:
def memory_report(df: pd.DataFrame) -> pd.DataFrame:
    """Detailed memory report per column."""
    mem = df.memory_usage(deep=True).drop('Index')
    report = pd.DataFrame({
        'dtype':    df.dtypes,
        'memory_MB': mem / 1024**2,
        'n_unique':  df.nunique(),
        'pct_unique': (df.nunique() / len(df) * 100).round(1)
    })
    report['memory_MB'] = report['memory_MB'].round(3)
    return report.sort_values('memory_MB', ascending=False)

print(memory_report(df_opt))

## 7. Auto Optimize Function — Production Ready

In [ ]:
def optimize_dataframe(df: pd.DataFrame, category_threshold: float = 0.5) -> pd.DataFrame:
    """
    Reduce DataFrame memory footprint:
    - Downcast integers and floats
    - Convert low-cardinality object columns to category
    """
    df = df.copy()
    before = df.memory_usage(deep=True).sum()

    for col in df.columns:
        dtype = df[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast='integer')

        elif pd.api.types.is_float_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast='float')

        elif dtype == object:
            cardinality = df[col].nunique() / len(df)
            if cardinality < category_threshold:
                df[col] = df[col].astype('category')

    after = df.memory_usage(deep=True).sum()
    print(f"Before: {before/1024**2:.1f} MB  →  After: {after/1024**2:.1f} MB  "
          f"(saved {(1 - after/before):.1%})")
    return df


df_auto = optimize_dataframe(df_naive)
print(df_auto.dtypes)

## 8. Parquet — Best Format for Large Files

In [ ]:
parquet_path = os.path.join(TMPDIR, 'transactions.parquet')

try:
    # Write optimized parquet
    df_opt.to_parquet(parquet_path, engine='pyarrow', compression='snappy', index=False)

    sizes = {
        'CSV (naive)':    os.path.getsize(csv_path) / 1024**2,
        'Parquet (snappy)': os.path.getsize(parquet_path) / 1024**2,
    }

    # Try gzip compression
    pq_gz = os.path.join(TMPDIR, 'transactions_gz.parquet')
    df_opt.to_parquet(pq_gz, engine='pyarrow', compression='gzip', index=False)
    sizes['Parquet (gzip)'] = os.path.getsize(pq_gz) / 1024**2

    for label, size in sizes.items():
        print(f"  {label:<22} {size:.1f} MB")

except ImportError:
    print("pyarrow not installed. Install: pip install pyarrow")

## 9. nrows + skiprows — Sampling a Large File

In [ ]:
# Read only 1000 rows for quick exploration
sample = pd.read_csv(csv_path, nrows=1000)
print(f"Sample shape: {sample.shape}")

# Skip rows 1-100000 (read rows 100001 onwards)
# skiprows can be an int (skip first N), list (skip specific rows), or callable
skip_first_half = pd.read_csv(
    csv_path,
    skiprows=range(1, 250_001),  # skip rows 1..250000 (0=header)
    header=0,
    names=big_df.columns.tolist()  # re-apply header
)
print(f"Skip first half: {skip_first_half.shape}")

## 10. Arrow Backend — Future of Pandas Performance

In [ ]:
try:
    # Arrow backend: faster string ops, lower memory, integrates with Polars/DuckDB
    df_arrow = pd.read_csv(
        csv_path,
        nrows=50_000,
        dtype_backend='pyarrow'
    )
    mem_arrow = df_arrow.memory_usage(deep=True).sum() / 1024**2
    print(f"Arrow backend memory (50k rows): {mem_arrow:.1f} MB")
    print(df_arrow.dtypes)
    
    # String operations on Arrow-backed series are faster
    import time
    
    df_obj = pd.read_csv(csv_path, nrows=50_000)
    
    start = time.perf_counter()
    _ = df_obj['category'].str.upper()
    t_obj = time.perf_counter() - start
    
    start = time.perf_counter()
    _ = df_arrow['category'].str.upper()
    t_arrow = time.perf_counter() - start
    
    print(f"\nstr.upper() — object: {t_obj*1000:.1f}ms | arrow: {t_arrow*1000:.1f}ms")

except (ImportError, Exception) as e:
    print(f"Arrow backend not available: {e}")
    print("Install: pip install pyarrow")

## 11. Real-World Pipeline — Large File ETL Pattern

In [ ]:
def process_transactions_chunked(
    filepath: str,
    chunk_size: int = 50_000,
    min_amount: float = 0
) -> pd.DataFrame:
    """
    Production pattern for large CSV processing:
    - Read in chunks
    - Filter in each chunk
    - Aggregate across chunks
    """
    dtype_spec = {
        'transaction_id': 'int32',
        'customer_id':    'int32',
        'category':       'category',
        'amount':         'float32',
        'quantity':       'int8',
        'city':           'category',
        'is_returned':    'int8',
    }

    summaries = []

    for chunk in pd.read_csv(filepath, chunksize=chunk_size, dtype=dtype_spec):
        # Filter
        valid = chunk[
            (chunk['amount'] > min_amount) &
            (chunk['is_returned'] == 0)
        ]

        if len(valid) == 0:
            continue

        # Aggregate this chunk
        chunk_summary = (
            valid
            .groupby(['category', 'city'], observed=True)
            .agg(revenue=('amount', 'sum'), orders=('transaction_id', 'count'))
            .reset_index()
        )
        summaries.append(chunk_summary)

    # Combine all chunk summaries
    combined = (
        pd.concat(summaries, ignore_index=True)
        .groupby(['category', 'city'], observed=True)
        .agg(revenue=('revenue', 'sum'), orders=('orders', 'sum'))
        .reset_index()
        .sort_values('revenue', ascending=False)
    )
    return combined


start = time.perf_counter()
summary = process_transactions_chunked(csv_path, chunk_size=50_000, min_amount=100)
print(f"Processed in {time.perf_counter() - start:.2f}s")
print(summary.head(10))

## 12. Quick Reference — Large File Strategies

| Strategy | When to Use |
|----------|-------------|
| `usecols=` | Read only needed columns |
| `dtype=` dict | Prevent dtype-inference overhead |
| `chunksize=` | File too large for RAM |
| `nrows=` | Quick exploration |
| `category` dtype | Low-cardinality string columns |
| Parquet format | Best for recurring analysis |
| PyArrow backend | Fastest for string-heavy data |
| Downcast numerics | 40-75% memory reduction on numerics |